# Figure 3 — are the loops really all short

Direct answer to that question. The three ECLs are peptide-scale; the N-terminus is not, and spans everything from short stubs to the LRR ectodomains of LGR4/5/6 and the glycoprotein hormone receptors. Signal peptides have already been trimmed upstream, so these are mature-receptor lengths.

## Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import style

style.apply()
if not style.available():
    print("Arial not found, matplotlib fell back to DejaVu Sans")

RESULTS, FIGS = ROOT / "results", ROOT / "figures"
FIGS.mkdir(exist_ok=True)
PAL = style.PALETTE
SEGMENTS = ["N-term", "ECL1", "ECL2", "ECL3"]

## Load

In [ ]:
seg = pd.read_csv(RESULTS / "segments_long.csv")
MARKS = [10, 25, 50, 100]

seg[seg.segment.isin(SEGMENTS)].groupby("segment")["length"].describe()[
    ["count", "min", "25%", "50%", "75%", "max"]].loc[SEGMENTS]

## Panels

In [ ]:
fig, ax = plt.subplot_mosaic([["hist", "hist"], ["ecdf", "frac"]],
                             figsize=(10.5, 7.5), constrained_layout=True)

a = ax["hist"]
bins = np.logspace(0, np.log10(seg["length"].max() * 1.1), 45)
for s in SEGMENTS:
    a.hist(seg.loc[seg.segment == s, "length"], bins=bins, alpha=0.5,
           color=PAL[s], label=s)
for m in MARKS:
    a.axvline(m, color="0.6", lw=0.8, ls=":")
    a.annotate(str(m), (m, a.get_ylim()[1]), fontsize=7, color="0.4",
               ha="center", va="bottom")
a.set_xscale("log")
a.set_xlabel("length (aa)")
a.set_ylabel("receptors")
a.set_title("Length distribution")
a.legend()

a = ax["ecdf"]
for s in SEGMENTS:
    x = np.sort(seg.loc[seg.segment == s, "length"].dropna().values)
    a.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=PAL[s], label=s)
for m in MARKS:
    a.axvline(m, color="0.6", lw=0.8, ls=":")
a.set_xscale("log")
a.set_xlabel("length (aa)")
a.set_ylabel("cumulative fraction")
a.set_title("What fraction is short")
a.legend()

a = ax["frac"]
w = 0.8 / len(SEGMENTS)
for i, s in enumerate(SEGMENTS):
    x = seg.loc[seg.segment == s, "length"].dropna().values
    a.bar(np.arange(len(MARKS)) + i * w, [(x <= m).mean() * 100 for m in MARKS],
          width=w, color=PAL[s], alpha=0.85, label=s)
a.set_xticks(np.arange(len(MARKS)) + 0.4 - w / 2)
a.set_xticklabels([f"<={m}" for m in MARKS])
a.set_ylabel("% of receptors")
a.set_ylim(0, 100)
a.set_title("Fraction at or below each length")
a.legend(fontsize=7)

fig.savefig(FIGS / "03_lengths.png")